# make_network — edit/*.shp + line_waits → network/{nodes,links}.parquet

QGIS 에서 편집한 `edit/*.shp` 와 노선별 배차간격 (`line_waits.parquet`) 을 받아,
**일련번호 node 와 (fromNode, toNode, timeFT, timeTF) link 형태의 그래프**로 출력.

**보행망은 포함되지 않는다** — node 는 station 뿐, link 는 station 간 subway/transfer 만.
보행망 결합은 받은 쪽에서 별도로 처리.

## 입력
- `edit/{stations,lines,transfer}.shp` — QGIS 편집 원본
- `line_waits.parquet` — 노선별 배차간격 (초). 환승 link 의 대기시간 누적에 사용.
- `opening.tsv` (선택) — 그 max date 이후 begin 인 행 제외.

## 출력 (`network/`)
- `nodes.parquet` — `id (0..N-1), linenm, statnm, x_5179, y_5179, lng, lat, begin, effective_begin`
- `links.parquet` — `id, fromNode, toNode, timeFT, timeTF, kind, begin, linenm_from, linenm_to, length_m, geometry`

## 시간 정책
- **subway (lines) link**: 양방향 동일 = `lines.time` (정차/통과 포함)
- **transfer link**: 비대칭. `timeFT = transfer.time + line_waits[to노선]`, `timeTF = transfer.time + line_waits[from노선]`.
  즉 link 진입 시 도착 노선의 배차간격(/2 의 평균) 을 더해줘서 대기시간이 자연스럽게 누적되게 만든 구조.

In [1]:
# cell 1 — setup
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point, LineString

HERE = Path('.').resolve()
EDIT = HERE / 'edit'
NETWORK = HERE / 'network'
NETWORK.mkdir(exist_ok=True)
print('EDIT:', EDIT)
print('NETWORK:', NETWORK)

EDIT: Z:\Github\seoulsubway2\backend\export\subway_network\edit
NETWORK: Z:\Github\seoulsubway2\backend\export\subway_network\network


In [2]:
# cell 2 — stations 로드 + 정규화
# eff_begin (shp 10자 제한) → effective_begin 복원, 보조 컬럼 제외.
sta_e = gpd.read_file(EDIT / 'stations.shp', encoding='utf-8')
if 'eff_begin' in sta_e.columns:
    sta_e = sta_e.rename(columns={'eff_begin': 'effective_begin'})
expected = ['begin', 'effective_begin', 'linenm', 'statnm', 'geometry']
if 'effective_begin' not in sta_e.columns:
    sta_e['effective_begin'] = ''
missing = [c for c in expected if c not in sta_e.columns]
assert not missing, f'stations.shp missing cols: {missing}'
stations = sta_e[expected].copy()
if not pd.api.types.is_string_dtype(stations['begin']):
    stations['begin'] = pd.to_datetime(stations['begin']).dt.strftime('%Y-%m-%d')
if not pd.api.types.is_string_dtype(stations['effective_begin']):
    stations['effective_begin'] = pd.to_datetime(stations['effective_begin'], errors='coerce').dt.strftime('%Y-%m-%d')
stations['effective_begin'] = stations['effective_begin'].fillna('').astype(str)
stations.loc[stations['effective_begin'].isin(['NaT', 'nan', 'None']), 'effective_begin'] = ''
stations['linenm'] = stations['linenm'].astype(str)
stations['statnm'] = stations['statnm'].astype(str)
stations = gpd.GeoDataFrame(stations, geometry='geometry', crs=sta_e.crs)
print('stations (raw):', stations.shape, 'crs=', stations.crs.to_authority() if stations.crs else None)
stations.head()

stations (raw): (1088, 5) crs= ('EPSG', '5179')


,begin,effective_begin,linenm,statnm,geometry
0,2016-02-27,,수인선,인천,POINT (921961.307 1942281.998)
1,2016-02-27,,수인선,신포,POINT (922592.904 1941382.16)
2,2016-02-27,,수인선,숭의,POINT (923795.045 1940527.716)
3,2016-02-27,,수인선,인하대,POINT (924798.872 1939130.672)
4,2012-06-30,,수인선,송도,POINT (925221.055 1937074.466)


In [3]:
# cell 3 — opening.tsv 의 max date 를 cutoff 로 사용.
# 그보다 미래 begin 행은 미정 노선으로 간주해 제외.
OPENING_TSV = HERE / 'opening.tsv'
if OPENING_TSV.exists():
    _op = pd.read_csv(OPENING_TSV, sep='\t')
    CUTOFF = _op['date'].max()
    print(f'opening.tsv max: {CUTOFF}  (rows={len(_op)})')
else:
    CUTOFF = '2099-12-31'
    print(f'opening.tsv not found → CUTOFF={CUTOFF} (all rows kept)')

before = len(stations)
stations = stations[stations['begin'] <= CUTOFF].reset_index(drop=True)
print(f'stations: {before} -> {len(stations)} (cutoff <= {CUTOFF})')

opening.tsv max: 2032-12-31  (rows=24)
stations: 1088 -> 915 (cutoff <= 2032-12-31)


In [4]:
# cell 4 — lines 로드 + 정규화 + cutoff 적용
ln_e = gpd.read_file(EDIT / 'lines.shp', encoding='utf-8')
expected = ['linenm', 'begin', 'time', 'geometry']
missing = [c for c in expected if c not in ln_e.columns]
assert not missing, f'lines.shp missing cols: {missing}'
lines = ln_e[expected].copy()
if not pd.api.types.is_string_dtype(lines['begin']):
    lines['begin'] = pd.to_datetime(lines['begin']).dt.strftime('%Y-%m-%d')
lines['linenm'] = lines['linenm'].astype(str)
lines['time'] = lines['time'].astype('float64')
lines = gpd.GeoDataFrame(lines, geometry='geometry', crs=ln_e.crs)
print('lines (raw):', lines.shape)

before = len(lines)
lines = lines[lines['begin'].notna() & (lines['begin'] <= CUTOFF)].reset_index(drop=True)
print(f'lines: {before} -> {len(lines)} (cutoff <= {CUTOFF})')
lines.head()

lines (raw): (1045, 4)
lines: 1045 -> 877 (cutoff <= 2032-12-31)


,linenm,begin,time,geometry
0,서울7호선,2012-10-27,100.0,"LINESTRING (932064.785 1945582.535, 932957.649..."
1,서울7호선,2012-10-27,110.0,"LINESTRING (934953.717 1945294.078, 936078.579..."
2,서울2호선,1982-12-23,90.0,"LINESTRING (958284.166 1944421.977, 959019.802..."
3,서울8호선,1996-11-23,100.0,"LINESTRING (965744.972 1944536.191, 965282.946..."
4,서울8호선,1996-11-23,110.0,"LINESTRING (966245.832 1943833.202, 965744.972..."


In [5]:
# cell 5 — transfer 로드
tr_e = gpd.read_file(EDIT / 'transfer.shp', encoding='utf-8')
expected = ['time', 'geometry']
missing = [c for c in expected if c not in tr_e.columns]
assert not missing, f'transfer.shp missing cols: {missing}'
transfer = tr_e[expected].copy()
transfer['time'] = transfer['time'].astype('float64')
transfer = gpd.GeoDataFrame(transfer, geometry='geometry', crs=tr_e.crs)
print('transfer (raw):', transfer.shape)
transfer.head()

transfer (raw): (459, 2)


,time,geometry
0,130.0,"LINESTRING (955320.657 1951458.423, 955372.399..."
1,130.0,"LINESTRING (962851.609 1950934.57, 962885.748 ..."
2,300.0,"LINESTRING (949115.429 1950995.587, 949386.722..."
3,240.0,"LINESTRING (949115.429 1950995.587, 948899.976..."
4,210.0,"LINESTRING (949115.429 1950995.587, 949401.442..."


In [6]:
# cell 6 — transfer 의 시작/끝 점이 모두 stations 의 어느 한 점과 1m 이내로 매칭돼야 유효.
# 한쪽이라도 매칭 실패면 해당 행 제거.
TOL_M = 1.0

starts = gpd.GeoDataFrame(
    {'tid': range(len(transfer))},
    geometry=[Point(g.coords[0])  for g in transfer.geometry],
    crs=transfer.crs,
)
ends = gpd.GeoDataFrame(
    {'tid': range(len(transfer))},
    geometry=[Point(g.coords[-1]) for g in transfer.geometry],
    crs=transfer.crs,
)
stations_geo = stations[['geometry']]

m_start = gpd.sjoin_nearest(starts, stations_geo, max_distance=TOL_M, how='inner')
m_end   = gpd.sjoin_nearest(ends,   stations_geo, max_distance=TOL_M, how='inner')

valid_tids = set(m_start['tid']) & set(m_end['tid'])
before = len(transfer)
transfer = transfer.iloc[sorted(valid_tids)].reset_index(drop=True)
print(f'transfer: {before} -> {len(transfer)} (양 끝점 모두 station 매칭, tol={TOL_M}m)')

transfer: 459 -> 317 (양 끝점 모두 station 매칭, tol=1.0m)


## (선택) 누락 환승 점검

In [7]:
# cell 7 — 동일 이름 station 페어가 현재 transfer 와 일치하는지 검증
from itertools import combinations

DIST_M = 1000.0

derived_pairs = set()
for name, g in stations.groupby('statnm'):
    if g['linenm'].nunique() < 2:
        continue
    idxs = g.index.tolist()
    for i, j in combinations(idxs, 2):
        if stations.geometry.loc[i].distance(stations.geometry.loc[j]) <= DIST_M:
            derived_pairs.add(frozenset([i, j]))

starts_pts = gpd.GeoDataFrame(
    {'tid': range(len(transfer))},
    geometry=[Point(g.coords[0])  for g in transfer.geometry], crs=transfer.crs,
)
ends_pts = gpd.GeoDataFrame(
    {'tid': range(len(transfer))},
    geometry=[Point(g.coords[-1]) for g in transfer.geometry], crs=transfer.crs,
)
stations_id = stations.assign(sid=stations.index)[['sid', 'geometry']]
ms = gpd.sjoin_nearest(starts_pts, stations_id, max_distance=1.0, how='inner').drop_duplicates('tid')
me = gpd.sjoin_nearest(ends_pts,   stations_id, max_distance=1.0, how='inner').drop_duplicates('tid')
start_map = dict(zip(ms['tid'], ms['sid']))
end_map   = dict(zip(me['tid'], me['sid']))
existing_pairs = {
    frozenset([start_map[t], end_map[t]])
    for t in range(len(transfer))
    if t in start_map and t in end_map and start_map[t] != end_map[t]
}
only_derived = derived_pairs - existing_pairs
only_existing = existing_pairs - derived_pairs
print(f'derived (이름같음, <={DIST_M:g}m): {len(derived_pairs)}, existing transfer: {len(existing_pairs)}')
print(f'derived 에만 (transfer 누락 가능): {len(only_derived)}')
print(f'existing 에만 (이름은 다른데 transfer 있음): {len(only_existing)}')

def show(label, pairs):
    print(f'\n[{label}] {len(pairs)}건')
    rows = []
    for pair in pairs:
        a, b = sorted(pair)
        ra, rb = stations.loc[a], stations.loc[b]
        d = stations.geometry.loc[a].distance(stations.geometry.loc[b])
        rows.append((ra['statnm'], ra['linenm'], rb['statnm'], rb['linenm'], round(d, 1)))
    rows.sort()
    for r in rows:
        print(f'  {r[0]:>10s}({r[1]:>10s}) <-> {r[2]:>10s}({r[3]:>10s})  {r[4]}m')

show('derived 에만 (이름 같고 <=1km, transfer 누락)', only_derived)
show('existing 에만 (transfer 있음, 이름 다름)', only_existing)

derived (이름같음, <=1000m): 329, existing transfer: 317
derived 에만 (transfer 누락 가능): 15
existing 에만 (이름은 다른데 transfer 있음): 3

[derived 에만 (이름 같고 <=1km, transfer 누락)] 15건
          곡산(     경의중앙선) <->         곡산(       서해선)  33.6m
          구성(       분당선) <->         구성(     GTX_A)  109.8m
          독산(      신안산선) <->         독산(     서울1호선)  850.9m
          백마(     경의중앙선) <->         백마(       서해선)  33.9m
         상록수(     서울4호선) <->        상록수(     GTX_C)  33.7m
          신촌(     경의중앙선) <->         신촌(     서울2호선)  660.8m
         왕십리(     경의중앙선) <->        왕십리(     GTX_C)  99.0m
         왕십리(       동북선) <->        왕십리(     GTX_C)  301.2m
         왕십리(       분당선) <->        왕십리(     GTX_C)  90.8m
         왕십리(     서울2호선) <->        왕십리(     GTX_C)  126.2m
         왕십리(     서울5호선) <->        왕십리(     GTX_C)  173.2m
         인덕원(     서울4호선) <->        인덕원(     GTX_C)  15.9m
          일산(     경의중앙선) <->         일산(       서해선)  36.6m
      정부과천청사(     서울4호선) <->     정부과천청사(     GTX_C)  924.1m


In [8]:
# cell 8 — 누락된 환승 수동 추가.
# 위 cell 7 의 'derived 에만' 출력을 참고해 (statnm, time_seconds) 튜플로 추가.
# 같은 statnm 이 정확히 2개 있을 때만 동작 (그 둘을 직선으로 잇는 transfer 추가).
EXTRA_TRANSFERS = []
# 예: EXTRA_TRANSFERS = [('곡산', 10), ('백마', 10), ('일산', 10), ('풍산', 10), ('구성', 300)]

if EXTRA_TRANSFERS:
    new_rows = []
    for statnm, t in EXTRA_TRANSFERS:
        pts = stations.loc[stations['statnm'] == statnm, 'geometry'].tolist()
        assert len(pts) == 2, f'{statnm}: 기대 2개, 실제 {len(pts)}개'
        new_rows.append({'time': float(t), 'geometry': LineString([pts[0], pts[1]])})
    extra = gpd.GeoDataFrame(new_rows, crs=transfer.crs)
    before = len(transfer)
    transfer = pd.concat([transfer, extra], ignore_index=True)
    print(f'transfer: {before} -> {len(transfer)} (수동 추가 {len(extra)}건)')
else:
    print('EXTRA_TRANSFERS 비어있음 — 추가 환승 없음')

EXTRA_TRANSFERS 비어있음 — 추가 환승 없음


## node / link 그래프 빌드

- node id = `stations.index` (0..N-1).
- subway link: lines 의 양 끝점을 stations 와 1m 이내 매칭 → (from, to, time).
- transfer link: 마찬가지로 매칭. timeFT/TF 에 line_waits 의 배차간격을 더해 비대칭으로 둠.

In [9]:
# cell 9 — stations 에 node id (0..N-1) 부여
stations = stations.reset_index(drop=True)
stations['id'] = stations.index.astype(np.int32)
print('nodes:', len(stations))

nodes: 915


In [10]:
# cell 10 — line_waits 로드
WAITS_PATH = HERE / 'line_waits.parquet'
if WAITS_PATH.exists():
    line_waits = pd.read_parquet(WAITS_PATH)
    wait_map = dict(zip(line_waits['linenm'], line_waits['waittm']))
    print(f'line_waits: {len(line_waits)} 노선')
else:
    wait_map = {}
    print('line_waits.parquet 없음 — 모든 노선 대기시간 0초로 처리')

# stations 의 linenm 중 line_waits 에 없는 노선 경고
missing = sorted(set(stations['linenm'].unique()) - set(wait_map.keys()))
if missing:
    print(f'WARNING: 대기시간 미정 노선 (0초 적용): {missing}')

line_waits: 42 노선


In [11]:
# cell 11 — lines / transfer 의 양 끝점을 stations 와 매칭 (sjoin_nearest, 1m)
def endpoints_to_sid(geoms, stations_pts, max_dist=1.0):
    starts = gpd.GeoDataFrame(
        {'i': range(len(geoms))},
        geometry=[Point(g.coords[0])  for g in geoms], crs=stations_pts.crs,
    )
    ends = gpd.GeoDataFrame(
        {'i': range(len(geoms))},
        geometry=[Point(g.coords[-1]) for g in geoms], crs=stations_pts.crs,
    )
    s_match = gpd.sjoin_nearest(starts, stations_pts, max_distance=max_dist, how='left')
    e_match = gpd.sjoin_nearest(ends,   stations_pts, max_distance=max_dist, how='left')
    s_match = s_match.drop_duplicates('i', keep='first').set_index('i')['sid']
    e_match = e_match.drop_duplicates('i', keep='first').set_index('i')['sid']
    return s_match, e_match

stations_pts = stations[['geometry']].copy()
stations_pts['sid'] = stations['id']

lines_u, lines_v = endpoints_to_sid(lines.geometry.tolist(), stations_pts)
trans_u, trans_v = endpoints_to_sid(transfer.geometry.tolist(), stations_pts)
print(f'lines:     matched {lines_u.notna().sum()}/{len(lines)} starts, {lines_v.notna().sum()}/{len(lines)} ends')
print(f'transfer:  matched {trans_u.notna().sum()}/{len(transfer)} starts, {trans_v.notna().sum()}/{len(transfer)} ends')

lines:     matched 875/877 starts, 877/877 ends
transfer:  matched 317/317 starts, 317/317 ends


In [12]:
# cell 12 — link 테이블 구성
sid_to_linenm = dict(zip(stations['id'], stations['linenm']))
sid_to_begin  = dict(zip(stations['id'], stations['begin']))

rows = []

# (a) subway (lines) — 양방향 동일 = lines.time
for i in range(len(lines)):
    su, sv = lines_u.get(i), lines_v.get(i)
    if pd.isna(su) or pd.isna(sv):
        continue
    su, sv = int(su), int(sv)
    if su == sv:
        continue
    t = float(lines['time'].iloc[i])
    rows.append({
        'fromNode': su, 'toNode': sv,
        'timeFT': t, 'timeTF': t,
        'kind': 'subway',
        'begin': lines['begin'].iloc[i],
        'linenm_from': lines['linenm'].iloc[i],
        'linenm_to':   lines['linenm'].iloc[i],
        'length_m': float(lines.geometry.iloc[i].length),
        'geometry':  lines.geometry.iloc[i],
    })

# (b) transfer — 비대칭. timeFT = transfer.time + wait[to노선], timeTF = transfer.time + wait[from노선]
for i in range(len(transfer)):
    su, sv = trans_u.get(i), trans_v.get(i)
    if pd.isna(su) or pd.isna(sv):
        continue
    su, sv = int(su), int(sv)
    if su == sv:
        continue
    t = float(transfer['time'].iloc[i])
    line_from = sid_to_linenm.get(su)
    line_to   = sid_to_linenm.get(sv)
    wait_to   = float(wait_map.get(line_to,   0))
    wait_from = float(wait_map.get(line_from, 0))
    # transfer 의 begin = max(from역 begin, to역 begin)
    bf = sid_to_begin.get(su, '')
    bt = sid_to_begin.get(sv, '')
    if not bf:
        b = bt
    elif not bt:
        b = bf
    else:
        b = max(bf, bt)
    rows.append({
        'fromNode': su, 'toNode': sv,
        'timeFT': t + wait_to, 'timeTF': t + wait_from,
        'kind': 'transfer',
        'begin': b,
        'linenm_from': line_from, 'linenm_to': line_to,
        'length_m': float(transfer.geometry.iloc[i].length),
        'geometry': transfer.geometry.iloc[i],
    })

links = gpd.GeoDataFrame(rows, geometry='geometry', crs=stations.crs)
links = links.sort_values(['kind', 'begin'], kind='stable').reset_index(drop=True)
links.insert(0, 'id', np.arange(len(links), dtype=np.int32))
print(f'links: {len(links)}  (subway={(links["kind"]=="subway").sum()}, '
      f'transfer={(links["kind"]=="transfer").sum()})')
links.head()

links: 1192  (subway=875, transfer=317)


,id,fromNode,toNode,timeFT,timeTF,kind,begin,linenm_from,linenm_to,length_m,geometry
0,0,418,419,120.0,120.0,subway,1974-08-15,서울1호선,서울1호선,1162.450337,"LINESTRING (946371.746 1939831.203, 946007.308..."
1,1,419,420,150.0,150.0,subway,1974-08-15,서울1호선,서울1호선,1824.333982,"LINESTRING (946007.308 1940935.048, 945415.278..."
2,2,332,333,120.0,120.0,subway,1974-08-15,서울1호선,서울1호선,1108.343890,"LINESTRING (932867.798 1943527.845, 933971.469..."
3,3,417,418,180.0,180.0,subway,1974-08-15,서울1호선,서울1호선,2442.523388,"LINESTRING (947123.508 1937511.362, 947071.493..."
4,4,411,412,120.0,120.0,subway,1974-08-15,서울1호선,서울1호선,1172.433527,"LINESTRING (951136.505 1927321.977, 951159.176..."


In [13]:
# cell 13 — node 테이블 (id, linenm, statnm, x_5179, y_5179, lng, lat, begin, effective_begin, geometry)
pts_4326 = stations.geometry.to_crs(4326)
nodes = gpd.GeoDataFrame({
    'id':        stations['id'].astype(np.int32),
    'linenm':    stations['linenm'],
    'statnm':    stations['statnm'],
    'x_5179':    stations.geometry.x.astype(np.float64),
    'y_5179':    stations.geometry.y.astype(np.float64),
    'lng':       pts_4326.x.astype(np.float64),
    'lat':       pts_4326.y.astype(np.float64),
    'begin':            stations['begin'],
    'effective_begin':  stations['effective_begin'],
    'geometry':  stations.geometry,
}, geometry='geometry', crs=stations.crs)
print('nodes:', len(nodes))
nodes.head()

nodes: 915


,id,linenm,statnm,x_5179,y_5179,lng,lat,begin,effective_begin,geometry
0,0,수인선,인천,921961.3068,1942281.998,126.617396,37.476474,2016-02-27,,POINT (921961.307 1942281.998)
1,1,수인선,신포,922592.9043,1941382.160,126.624633,37.468417,2016-02-27,,POINT (922592.904 1941382.16)
2,2,수인선,숭의,923795.0453,1940527.716,126.638315,37.460816,2016-02-27,,POINT (923795.045 1940527.716)
3,3,수인선,인하대,924798.8725,1939130.672,126.649807,37.448307,2016-02-27,,POINT (924798.872 1939130.672)
4,4,수인선,송도,925221.0553,1937074.466,126.654788,37.429809,2012-06-30,,POINT (925221.055 1937074.466)


In [14]:
# cell 14 — 저장 (parquet + tsv 둘 다)
# tsv 는 geometry 컬럼을 WKT 문자열로 직렬화. 다른 도구 (excel, R, 다른 언어) 에서 읽기 편함.
import pandas as pd

nodes.to_parquet(NETWORK / 'nodes.parquet', compression='zstd', compression_level=9, index=False)
links.to_parquet(NETWORK / 'links.parquet', compression='zstd', compression_level=9, index=False)
print(f'wrote {NETWORK / "nodes.parquet"}  ({len(nodes)} rows, '
      f'{(NETWORK / "nodes.parquet").stat().st_size/1024:.1f} KiB)')
print(f'wrote {NETWORK / "links.parquet"}  ({len(links)} rows, '
      f'{(NETWORK / "links.parquet").stat().st_size/1024:.1f} KiB)')

# tsv — geometry → WKT 문자열
nodes_tsv = pd.DataFrame(nodes.drop(columns=['geometry']))
nodes_tsv['geometry_wkt'] = nodes.geometry.to_wkt()
nodes_tsv.to_csv(NETWORK / 'nodes.tsv', sep='\t', index=False, encoding='utf-8')

links_tsv = pd.DataFrame(links.drop(columns=['geometry']))
links_tsv['geometry_wkt'] = links.geometry.to_wkt()
links_tsv.to_csv(NETWORK / 'links.tsv', sep='\t', index=False, encoding='utf-8')

print(f'wrote {NETWORK / "nodes.tsv"}  ({(NETWORK / "nodes.tsv").stat().st_size/1024:.1f} KiB)')
print(f'wrote {NETWORK / "links.tsv"}  ({(NETWORK / "links.tsv").stat().st_size/1024:.1f} KiB)')

# 검증 출력
print(f'\n=== 검증 ===')
print(f'nodes: {len(nodes)}, id range: [{nodes["id"].min()}, {nodes["id"].max()}]')
print(f'links: {len(links)}')
print(f'  subway:   {(links["kind"]=="subway").sum():4d}')
print(f'  transfer: {(links["kind"]=="transfer").sum():4d}')
asym = links[links['timeFT'] != links['timeTF']]
print(f'  asymmetric (FT != TF): {len(asym)}  (transfer 에서만 발생해야 정상)')
assert links['fromNode'].between(0, len(nodes)-1).all() and links['toNode'].between(0, len(nodes)-1).all(), \
    'link 가 nodes 범위 밖의 id 참조함'
print('OK — link 의 fromNode/toNode 모두 nodes.id 범위 안에 있음')

wrote Z:\Github\seoulsubway2\backend\export\subway_network\network\nodes.parquet  (915 rows, 62.4 KiB)


wrote Z:\Github\seoulsubway2\backend\export\subway_network\network\links.parquet  (1192 rows, 181.4 KiB)
wrote Z:\Github\seoulsubway2\backend\export\subway_network\network\nodes.tsv  (120.0 KiB)
wrote Z:\Github\seoulsubway2\backend\export\subway_network\network\links.tsv  (420.4 KiB)

=== 검증 ===
nodes: 915, id range: [0, 914]
links: 1192
  subway:    875
  transfer:  317
  asymmetric (FT != TF): 271  (transfer 에서만 발생해야 정상)
OK — link 의 fromNode/toNode 모두 nodes.id 범위 안에 있음
